# Don't classify. Hallucinate.

*Cheap LLM classification into a taxonomy too large to fit in a prompt — with Couchbase Vector Search doing the resolution step.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/01_hypothetical_classification.ipynb)

Classifying text into a controlled vocabulary is the most boring, most common LLM task in
search: given a query or a product title, which of our categories does it belong to?

The obvious approach is to put the vocabulary in the prompt and constrain the output to it.
That works until the vocabulary gets real. Wayfair's product taxonomy — the one in this
notebook — has **1,623 category paths**. Shipping it on every request costs tens of
thousands of tokens per call, and hits the size limits on structured-output schemas.

[Doug Turnbull's suggestion](https://softwaredoug.com/blog/2026/08/10/hypothetical-classifications):
**don't send the vocabulary at all.** Ask a small, cheap model to *invent* the category path
it would expect to exist, then snap that invented path to the nearest real one with an
embedding lookup. The hallucination is the point — it produces a string shaped like a
taxonomy path, which lands in roughly the right neighbourhood of embedding space, and a
vector index does the rest.

It's the same trick as [HyDE](https://arxiv.org/abs/2212.10496) for retrieval, pointed at
classification: generate a plausible fake, then retrieve against it.

**What this notebook does**

1. Loads the 1,623-path WANDS taxonomy and stores it in Couchbase with embeddings.
2. Builds a Search vector index over it.
3. Asks a small model to hallucinate a category path for a product — with none of the real
   taxonomy in the prompt.
4. Resolves the hallucination to a real path with a vector search.
5. **Measures it** against a direct-embedding baseline on held-out products with known
   categories.
6. Uses the resolved category as a filter on a product search, which is what you'd actually
   do with it.

**You need:** a Couchbase Capella cluster (free tier is fine — see
[`docs/capella-setup.md`](../docs/capella-setup.md)) and an API key for any OpenAI-compatible
model endpoint.

**Data:** [WANDS](https://github.com/wayfair/WANDS) (Wayfair, MIT licence) — 43k real product
listings filed under a real, messy retail taxonomy.

In [ ]:
# --- Setup. Works in a local checkout and on Colab. -------------------------
import os
import pathlib
import subprocess
import sys

# TODO: point at the repo before publishing, so the Colab badge and this cell work.
REPO_URL = os.environ.get("CBNB_REPO_URL", "https://github.com/OWNER/REPO")

try:
    import cbnb
except ModuleNotFoundError:
    here = pathlib.Path.cwd()
    root = next((p for p in [here, *here.parents] if (p / "cbnb" / "__init__.py").exists()), None)
    if root is None:
        # Colab: clone the repo so the committed datasets come with it.
        subprocess.check_call(["git", "clone", "--depth", "1", "--quiet", REPO_URL, "cbnb-repo"])
        root = pathlib.Path("cbnb-repo").resolve()
    sys.path.insert(0, str(root))
    import cbnb

# Installs anything missing, loads .env / Colab secrets, and reports what it found.
settings = cbnb.bootstrap(extras=["local-embeddings", "plots"])

## 1. The vocabulary problem

WANDS ships the category path every product was actually filed under. Here is the shape of it.

In [ ]:
from cbnb.datasets import load_wands_products, load_wands_taxonomy, Taxonomy

taxonomy = Taxonomy(load_wands_taxonomy())
products = load_wands_products()

print(f"{len(taxonomy):,} category paths across {len(taxonomy.departments())} departments")
print(f"{len(products):,} products in the committed sample\n")
for path in list(taxonomy)[300:305]:
    print(" ", path)

In [ ]:
# What would it cost to put the whole thing in the prompt?
vocabulary_text = "\n".join(taxonomy.paths)
approx_tokens = len(vocabulary_text) / 4  # rough, but the order of magnitude is the point

print(f"Full taxonomy: {len(vocabulary_text):,} characters ~= {approx_tokens:,.0f} tokens")
print(f"Per 10,000 classifications: ~{approx_tokens * 10_000 / 1e6:,.0f}M prompt tokens,")
print("before you have said a single word about the product you want classified.")

That's the tax on the obvious approach — paid on every single call, for a list that never
changes. Prompt caching helps, and a big enough context window makes it *possible*, but you
are still paying to re-read a static document to answer a five-word question.

So: don't send it.

## 2. Put the vocabulary in Couchbase instead

Each category path becomes a document with an embedding. The vocabulary lives in the
database, where a vector index can search it in a millisecond, instead of in the prompt,
where it costs money every time.

`Embedder` defaults to a local `all-MiniLM-L6-v2` — no API key, 384 dimensions, and it
encodes all 1,623 paths in a few seconds. Vectors come back L2-normalised, so the
`dot_product` similarity the index uses is a cosine similarity.

In [ ]:
from cbnb.embeddings import Embedder

embedder = Embedder()          # backend="api" to use your provider's /embeddings instead
print(embedder, "->", embedder.dims, "dimensions")

category_vectors = embedder.encode(taxonomy.paths, progress=True)
category_vectors.shape

In [ ]:
from cbnb.couchbase_io import connect, ensure_collection, upsert_docs

BUCKET = settings.cb_bucket
SCOPE = "hypothetical_classification"

cluster = connect(settings)
categories = ensure_collection(cluster, BUCKET, SCOPE, "categories")
print(f"Connected. Writing to {BUCKET}.{SCOPE}.categories")

In [ ]:
category_docs = {
    f"cat::{i:04d}": {
        "path": path,
        "leaf": taxonomy.leaf(path),
        "department": taxonomy.top_level(path),
        "depth": len(taxonomy.segments(path)),
        "embedding": category_vectors[i].tolist(),
    }
    for i, path in enumerate(taxonomy.paths)
}

upsert_docs(categories, category_docs)

### The vector index

A scope-level Search index with one `vector` field and a couple of `keyword` fields for
filtering. `wait_for_index` blocks until ingestion catches up — Search indexing is
asynchronous, and without the wait the first query in a fresh run quietly returns nothing.

In [ ]:
from cbnb.couchbase_io import ensure_vector_index, wait_for_index

CATEGORY_INDEX = "categories-vec"

ensure_vector_index(
    cluster,
    bucket_name=BUCKET, scope_name=SCOPE, collection_name="categories",
    index_name=CATEGORY_INDEX,
    vector_field="embedding", dims=embedder.dims,
    text_fields=["leaf"], keyword_fields=["path", "department"],
)
wait_for_index(cluster, bucket_name=BUCKET, scope_name=SCOPE,
               index_name=CATEGORY_INDEX, expected=len(category_docs))

## 3. Ask a small model to make something up

Now the interesting part. The prompt describes the *shape* of a category path — depth,
separator, capitalisation, the general register of a home-goods retailer — and says nothing
about which categories exist. The model is explicitly told to invent one.

Note what is not in this prompt: the taxonomy.

In [ ]:
from pydantic import BaseModel, Field
from cbnb.llm import LLM

llm = LLM()   # provider/model come from CBNB_LLM_PROVIDER / CBNB_LLM_MODEL
print(llm)


class HypotheticalCategory(BaseModel):
    """A category path the model believes a home-goods retailer would use."""
    department: str = Field(description="Top-level department, e.g. 'Furniture'")
    category_path: str = Field(
        description="Full path from department to leaf, joined by ' / ', 3-5 levels deep"
    )


CLASSIFY_SYSTEM = """\
You file products into the category taxonomy of a large home-goods retailer.

You do not have the taxonomy. Invent the category path you would expect it to contain.

Rules:
- 3 to 5 levels, most general first, joined by " / ".
- Level 1 is a department: Furniture, Outdoor, Lighting, Bed & Bath, Kitchen & Tabletop,
  Storage & Organization, Decor & Pillows, Appliances, Baby & Kids, Rugs, Pet, and so on.
- Each level narrows the one before it. The last level is the most specific.
- Use plural, title-cased retail category names ("Coffee Tables", not "a brown coffee table").
- Categories describe the kind of product, never its colour, material, brand or size.
- Output the path only. Do not explain."""


def imagine_category(product_name: str) -> HypotheticalCategory:
    return llm.structured(
        f"Product: {product_name}",
        HypotheticalCategory,
        system=CLASSIFY_SYSTEM,
    )


for name in ["brown coffee table",
             "solid wood platform bed",
             "outdoor propane fire pit table",
             "kids dinosaur night light"]:
    guess = imagine_category(name)
    print(f"{name:38s} -> {guess.category_path}")

Those paths are plausible and mostly **wrong** — they are not in the taxonomy. That is
fine. They are wrong in a very particular way: they are wrong the way a *category path* is
wrong, not the way a product title is wrong. Which makes them easy to fix.

## 4. Resolve the hallucination against Couchbase

One vector search per classification. The invented path is embedded and matched against the
1,623 real ones; the nearest neighbour is the answer.

In [ ]:
from cbnb.couchbase_io import vector_search


def resolve(text: str, k: int = 1):
    """Nearest real category path(s) to an arbitrary string."""
    return vector_search(
        cluster,
        bucket_name=BUCKET, scope_name=SCOPE, index_name=CATEGORY_INDEX,
        vector_field="embedding",
        query_vector=embedder.encode_one(text),
        k=k, fields=["path", "department"], num_candidates=max(k * 4, 20),
    )


def classify(product_name: str) -> dict:
    """Hallucinate, then resolve."""
    guess = imagine_category(product_name)
    hit = resolve(guess.category_path)[0]
    return {"product": product_name,
            "hallucinated": guess.category_path,
            "resolved": hit["path"],
            "score": hit.score}


import pandas as pd

demo = pd.DataFrame([classify(n) for n in [
    "brown coffee table",
    "solid wood platform bed",
    "outdoor propane fire pit table",
    "kids dinosaur night light",
    "stainless steel over the door towel rack",
]])
demo.style.hide(axis="index")

The invented `Furniture / Living Room / Tables / Coffee Tables` becomes the real
`Furniture / Living Room Furniture / Coffee Tables & End Tables / Coffee Tables`. The model
never saw either string.

## 5. Does it actually beat the obvious baseline?

The obvious baseline needs no LLM at all: embed the product title and find the nearest
category path. If that works as well, the LLM call is waste.

WANDS gives us ground truth — every product's real category path — so this is measurable.
Three numbers, because a hierarchical taxonomy makes "nearly right" meaningful:

- **exact** — the full path matches
- **department** — the top level matches
- **prefix overlap** — fraction of the true path's levels matched from the root

`N_EVAL` is the only knob that costs money. 150 products is enough to separate the two
methods; raise it if you want tighter error bars.

In [ ]:
N_EVAL = 150

sample = products.sample(N_EVAL, random_state=1729).reset_index(drop=True)
names = sample["product_name"].tolist()
truth = sample["category_hierarchy"].tolist()

# Baseline: embed the product title, take the nearest category. No LLM.
baseline = [hits[0]["path"] for hits in
            (resolve(n) for n in names)]

# Method: hallucinate a path, then resolve it. Concurrent, because 150 serial
# API calls is a coffee break.
guesses = llm.map(names, imagine_category, max_workers=8)
method = [resolve(g.category_path)[0]["path"] for g in guesses]

print("done")

In [ ]:
def score(predictions, label):
    exact = sum(p == t for p, t in zip(predictions, truth)) / len(truth)
    dept = sum(taxonomy.top_level(p) == taxonomy.top_level(t)
               for p, t in zip(predictions, truth)) / len(truth)
    overlap = sum(taxonomy.prefix_overlap(p, t) for p, t in zip(predictions, truth)) / len(truth)
    return {"method": label, "exact path": exact, "department": dept, "prefix overlap": overlap}


results = pd.DataFrame([
    score(baseline, "embed the title (no LLM)"),
    score(method, "hallucinate -> resolve"),
]).set_index("method")

results.style.format("{:.1%}").set_caption(f"n = {N_EVAL} WANDS products")

In [ ]:
import matplotlib.pyplot as plt

ax = results.plot.bar(rot=0, figsize=(8, 3.6), width=0.75,
                      color=["#ED2226", "#00B4D8", "#8E9AAF"])
ax.set_ylabel("accuracy")
ax.set_xlabel("")
ax.set_title(f"Classifying WANDS products into 1,623 categories (n={N_EVAL})")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.12))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

### Where each method goes wrong

Worth looking at, because the two methods fail differently.

In [ ]:
comparison = pd.DataFrame({
    "product": names,
    "true": truth,
    "baseline": baseline,
    "hallucinated": [g.category_path for g in guesses],
    "resolved": method,
})
comparison["baseline_ok"] = comparison["true"] == comparison["baseline"]
comparison["method_ok"] = comparison["true"] == comparison["resolved"]

fixed = comparison[comparison["method_ok"] & ~comparison["baseline_ok"]]
print(f"{len(fixed)} the LLM got right and the baseline did not:\n")
for _, row in fixed.head(4).iterrows():
    print(f"  {row['product']}")
    print(f"    baseline : {row['baseline']}")
    print(f"    imagined : {row['hallucinated']}")
    print(f"    resolved : {row['resolved']}  <- correct\n")

In [ ]:
broke = comparison[~comparison["method_ok"] & comparison["baseline_ok"]]
print(f"{len(broke)} the baseline got right and the LLM did not:\n")
for _, row in broke.head(4).iterrows():
    print(f"  {row['product']}")
    print(f"    true     : {row['true']}")
    print(f"    imagined : {row['hallucinated']}")
    print(f"    resolved : {row['resolved']}\n")

The pattern: embedding a product title directly puts you in *product* space, where "solid
wood platform bed" sits near other bed descriptions but not especially near the string
`Furniture / Bedroom Furniture / Beds & Headboards / Beds / Twin Beds`. The hallucinated
path is already in *taxonomy* space. The LLM's job is not to know the answer — it's to
translate into the right dialect.

Where it fails: products whose category depends on a detail the model reads past, and
retailer-specific idioms nobody would invent (`Accommodations / Guest Room Amenities / ...`
is Wayfair's word for the hospitality catalogue).

## 6. What it cost

In [ ]:
u = llm.usage
per_call = u.prompt_tokens / u.calls if u.calls else 0

print(f"Model    : {llm.provider.label} / {llm.model}")
print(f"Usage    : {u}")
if per_call:
    print(f"\nPer classification : ~{per_call:,.0f} prompt tokens")
    print(f"Taxonomy in prompt : ~{approx_tokens:,.0f} prompt tokens "
          f"({approx_tokens / per_call:,.0f}x more, on every single call)")
else:
    print("\nEvery call came from the on-disk cache. Re-run with "
          "LLM(cache=False) to measure real token usage.")

The vocabulary moved from a per-call prompt cost to a one-time indexing cost, and the
per-call cost dropped to a short prompt plus one vector search. Add a category and you
upsert one document — you don't touch the prompt, and you don't re-validate a schema.

## 7. What the classification is actually for

A category label is only worth having if it narrows a search. Load the products with their
own embeddings, then compare an unfiltered vector search against one prefiltered by the
department we just resolved.

In [ ]:
product_vectors = embedder.encode(products["product_name"].tolist(), progress=True)

product_collection = ensure_collection(cluster, BUCKET, SCOPE, "products")
upsert_docs(product_collection, {
    f"prod::{row.product_id}": {
        "product_id": int(row.product_id),
        "name": row.product_name,
        "category_path": row.category_hierarchy,
        "department": taxonomy.top_level(row.category_hierarchy),
        "embedding": product_vectors[i].tolist(),
    }
    for i, row in enumerate(products.itertuples())
})

PRODUCT_INDEX = "products-vec"
ensure_vector_index(
    cluster,
    bucket_name=BUCKET, scope_name=SCOPE, collection_name="products",
    index_name=PRODUCT_INDEX,
    vector_field="embedding", dims=embedder.dims,
    text_fields=["name"], keyword_fields=["category_path", "department"],
)
wait_for_index(cluster, bucket_name=BUCKET, scope_name=SCOPE,
               index_name=PRODUCT_INDEX, expected=len(products))

In [ ]:
import couchbase.search as search

QUERY = "something to keep drinks on next to the sofa"

guess = imagine_category(QUERY)
resolved_category = resolve(guess.category_path)[0]
department = resolved_category["department"]

print(f"query      : {QUERY}")
print(f"imagined   : {guess.category_path}")
print(f"resolved   : {resolved_category['path']}")
print(f"department : {department}\n")

query_vector = embedder.encode_one(QUERY)


def product_search(prefilter=None, k=5):
    return vector_search(
        cluster,
        bucket_name=BUCKET, scope_name=SCOPE, index_name=PRODUCT_INDEX,
        vector_field="embedding", query_vector=query_vector,
        k=k, fields=["name", "category_path"], num_candidates=100,
        prefilter=prefilter,
    )


unfiltered = product_search()
filtered = product_search(prefilter=search.TermQuery(department, field="department"))

pd.DataFrame({
    "unfiltered": [h["name"] for h in unfiltered],
    "filtered to " + department: [h["name"] for h in filtered],
})

Same query vector, same index, one extra clause — and the results stop wandering into
other departments. The classification is what made the filter possible, and it cost one
cheap LLM call and one vector search.

## Where to take this

- **Swap in a smaller model.** The whole premise is that this works without a frontier
  model. Set `CBNB_LLM_MODEL` to the cheapest thing your provider offers and re-run
  section 5 — the interesting question is where the accuracy actually falls off.
- **Hallucinate several paths and vote.** Sample the model three times at a higher
  temperature, resolve each, and take the most common real category.
- **Rerank instead of top-1.** Retrieve the 20 nearest real paths and let the model pick.
  Costs a second call, but the choice is now grounded — and 20 paths in a prompt is cheap
  where 1,623 was not.
- **Attribute extraction.** The same hallucinate-then-resolve pattern works for colours,
  materials and brands — any controlled vocabulary too big to enumerate.
- **Cache the resolutions.** Store the hallucinated path alongside its resolved category
  in Couchbase and the second occurrence of a near-identical product skips the LLM entirely.

Credit where it's due: the technique is
[Doug Turnbull's](https://softwaredoug.com/blog/2026/08/10/hypothetical-classifications),
and his [`cheat-at-search`](https://github.com/softwaredoug/cheat-at-search) repository has
the original implementation.

In [ ]:
# Clean up everything this notebook created. Uncomment to run.
# from cbnb.couchbase_io import drop_demo_data
# drop_demo_data(cluster, BUCKET, SCOPE)